# 15c. Kiểm tra số baseline trong compare_val và dựng lại bảng so sánh từ file trong repo (task 5.7, GĐ F)

Ở notebook 15, `compare_val_nf_ton_iot_v3.csv` cho baseline ToN cao hơn số đã lưu trong repo (LightGBM 0,935 so với 0,898). Notebook này: (1) so file `results_baselines_*.csv` do Kaggle mount (output của 05, 13) với bản trong repo, bằng kích thước, md5 và số trung bình; (2) dựng lại `compare_val_<bộ>.csv` chỉ từ các file kết quả đã commit trong repo (baseline 05, 13; GNN 07, 14; hybrid 15, 15b nếu có).

CPU, không dữ liệu gốc, không train. Gắn output của 05, 07, 13, 14. Bật Internet.

In [ ]:
import glob, hashlib, json, os, subprocess, sys, time
import pandas as pd

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
INFO = {"device": "CPU"}
T0 = time.time()


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-6000:], end="")
    print(r.stderr[-2000:], end="", file=sys.stderr)
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


run(["rm", "-rf", REPO])
run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", INFO["commit"])

## 1. File mount của Kaggle so với bản trong repo

In [ ]:
KAG = f"{REPO}/reports/kaggle"
PAIRS = {
    "nf_ton_iot_v3": ("gnn-iot-ids-05-baselines", "gnn-iot-ids-07-train-gnn"),
    "nf_bot_iot_v3": ("gnn-iot-ids-13-baselines", "gnn-iot-ids-14-train-gnn-bot"),
}


def md5(p):
    return hashlib.md5(open(p, "rb").read()).hexdigest()


def means(p):
    d = pd.read_csv(p)
    return d.groupby("model")["f1_macro"].mean().round(4).to_dict(), len(d)


check = {}
for ds, (kb, kg) in PAIRS.items():
    for kind, k, name in (("baseline", kb, f"results_baselines_{ds}.csv"), ("gnn", kg, f"results_gnn_{ds}.csv")):
        repo_p = f"{KAG}/{k}/{name}"
        mounted = sorted(glob.glob(f"/kaggle/input/**/{k}/{name}", recursive=True))
        e = {"repo": {"path": repo_p, "size": os.path.getsize(repo_p), "md5": md5(repo_p)}, "mounted": []}
        e["repo"]["f1_mean"], e["repo"]["rows"] = means(repo_p)
        for m in mounted:
            f1, rows = means(m)
            e["mounted"].append({"path": m, "size": os.path.getsize(m), "md5": md5(m), "rows": rows, "f1_mean": f1,
                                 "same_as_repo": md5(m) == e["repo"]["md5"]})
        check[f"{ds}|{kind}"] = e
        print(f"== {ds} {kind}: repo md5 {e['repo']['md5'][:8]} size {e['repo']['size']} rows {e['repo']['rows']}")
        for m in e["mounted"]:
            print(f"   mount {m['path']}\n     md5 {m['md5'][:8]} size {m['size']} rows {m['rows']} giống repo: {m['same_as_repo']}")
            if not m["same_as_repo"]:
                diff = {k2: (e["repo"]["f1_mean"].get(k2), v) for k2, v in m["f1_mean"].items() if e["repo"]["f1_mean"].get(k2) != v}
                print("     F1 trung bình khác (repo, mount):", diff)
json.dump(check, open(f"{WORK}/input_check.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

## 2. compare_val dựng lại chỉ từ file trong repo

In [ ]:
for ds, (kb, kg) in PAIRS.items():
    srcs = [f"{KAG}/{kb}/results_baselines_{ds}.csv", f"{KAG}/{kg}/results_gnn_{ds}.csv"]
    srcs += sorted(glob.glob(f"{KAG}/gnn-iot-ids-15*-hybrid*/results_hybrid*_{ds}.csv"))
    print("nguồn:", [s.replace(KAG + "/", "") for s in srcs])
    run([sys.executable, "-I", "scripts/compare_val.py", ds, f"{WORK}/compare_val_{ds}.csv", *srcs], cwd=REPO)

INFO["total_s"] = round(time.time() - T0, 1)
INFO["mount_equals_repo"] = {k: [m["same_as_repo"] for m in v["mounted"]] for k, v in check.items()}
json.dump(INFO, open(f"{WORK}/run_info.json", "w"), indent=1)
print(json.dumps(INFO, indent=1))